# AgroStake — Stage 1: Binary Gate Model Training
**TinyBinaryCNN** — Healthy vs. Diseased

This notebook trains the lightweight gate model that runs first on the ESP32-S3.
If the gate says **Healthy**, we stop and report immediately (fast path, ~60 ms).  
If the gate says **Diseased**, the full TinyAgCNN fires to identify the disease.

The dataset layout is identical to Phase 1/2 of AgroStake — you don't move anything.

```
grape-leaf-disease-augmented-dataset/
    train/
        Black_rot/
        Esca/
        Healthy/
        Leaf_blight/
    val/
        ...
```

Binary label mapping:
- `Healthy`  → **0**
- `Black_rot`, `Esca`, `Leaf_blight` → **1**

In [1]:
# ─── 0. Dependencies ──────────────────────────────────────────────────────────
# pip install torch torchvision onnx onnxruntime tqdm matplotlib

import os
import copy
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
# ─── 1. Configuration ─────────────────────────────────────────────────────────

# ↓ Same root as your AgroStake Phase 1 notebook
DATASET_ROOT = Path('../0_data/augmented-dataset')

IMG_SIZE     = 96        # must match ESP32 firmware
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
LR           = 3e-4
WEIGHT_DECAY = 1e-4

CHECKPOINT   = Path('./TinyBinaryCNN_v1_best.pth')
ONNX_OUT     = Path('./TinyBinaryCNN_v1.onnx')

# Healthy class name exactly as it appears in the folder
HEALTHY_FOLDER = 'Healthy'

assert DATASET_ROOT.exists(), f'Dataset not found at {DATASET_ROOT.resolve()}'

In [3]:
# ─── 2. Binary Label Wrapper ─────────────────────────────────────────────────
# We reuse the 4-class ImageFolder, then remap labels at batch time.
# 0 = Healthy, 1 = Diseased — no files need to be moved or copied.

class BinaryGrapeDataset(torch.utils.data.Dataset):
    """Wraps an ImageFolder and collapses 4 classes to 2."""

    def __init__(self, root: Path, transform=None):
        self.base = datasets.ImageFolder(root=str(root), transform=transform)
        # Find which integer index ImageFolder assigned to Healthy
        if HEALTHY_FOLDER not in self.base.class_to_idx:
            raise ValueError(
                f"'{HEALTHY_FOLDER}' not found in {root}. "
                f"Available: {list(self.base.class_to_idx.keys())}"
            )
        self.healthy_idx = self.base.class_to_idx[HEALTHY_FOLDER]
        print(f'  {root.name}: classes={self.base.classes}, '
              f'healthy_idx={self.healthy_idx}, n={len(self.base)}')

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, orig_label = self.base[idx]
        binary_label = 0 if orig_label == self.healthy_idx else 1
        return img, binary_label

    @property
    def targets(self):
        """Binary targets — used for class-balanced sampler."""
        return [
            0 if t == self.healthy_idx else 1
            for t in self.base.targets
        ]

In [4]:
# ─── 3. Transforms ───────────────────────────────────────────────────────────
# Identical normalisation to Phase 1/2 so the preprocessing pipeline is consistent.
# The ESP32 receives int8 = pixel - 128; that is equivalent to mean≈0.5 × 255.

IMAGENET_MEAN = [0.485, 0.456, 0.406]   # kept for augmentation stability
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load full training dataset, then split into train/val
full_train_ds = BinaryGrapeDataset(DATASET_ROOT / 'train', transform=train_tf)

# Split: 80% train, 20% validation (from training data)
train_size = int(0.8 * len(full_train_ds))
val_size   = len(full_train_ds) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_train_ds, [train_size, val_size])

# Load test dataset (separate, held-out evaluation set)
test_ds  = BinaryGrapeDataset(DATASET_ROOT / 'test',   transform=val_tf)

# ── Balanced sampler (healthy vs diseased may be imbalanced) ──────────────────
# Note: sampling from the training subset only
train_targets = train_ds.dataset.targets
train_indices = train_ds.indices
train_targets_subset = [train_targets[i] for i in train_indices]
class_counts  = np.bincount(train_targets_subset)
print(f'Train subset: Healthy={class_counts[0]}, Diseased={class_counts[1]}')
weights = 1.0 / class_counts[train_targets_subset]
sampler = WeightedRandomSampler(weights, num_samples=len(train_targets_subset), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')


  train: classes=['Black Rot', 'ESCA', 'Healthy', 'Leaf Blight'], healthy_idx=2, n=14444
  test: classes=['Black Rot', 'ESCA', 'Healthy', 'Leaf Blight'], healthy_idx=2, n=3610
Train subset: Healthy=2712, Diseased=8843
Train batches: 181 | Val batches: 46 | Test batches: 57


In [5]:
# ─── 4. TinyBinaryCNN Architecture ──────────────────────────────────────────
# Goal: < 150 KB quantised on the ESP32-S3
# Inference time target: < 70 ms
#
# Architecture (NHWC on device, NCHW in PyTorch):
#   Input  96×96×3
#   Conv1  8 filters 3×3 VALID → BN → ReLU → MaxPool2 → 47×47×8
#   Conv2  16 filters 3×3 VALID → BN → ReLU → MaxPool2 → 22×22×16
#   Flatten 7744
#   FC1    7744 → 64 → ReLU → Dropout(0.3)
#   FC2    64 → 2

class TinyBinaryCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  8, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=0, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # Compute flatten size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            flat  = self.features(dummy).flatten(1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = TinyBinaryCNN(num_classes=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}  (~{total_params * 4 / 1024:.1f} KB fp32)')
print(f'Estimated INT8 size: ~{total_params / 1024:.1f} KB')

Parameters: 497,226  (~1942.3 KB fp32)
Estimated INT8 size: ~485.6 KB


In [6]:
# ─── 5. Loss, Optimiser, Scheduler ──────────────────────────────────────────
# Cross-entropy is sufficient; we do NOT need knowledge distillation here
# because the binary task is much easier than 4-class disease ID.

criterion = nn.CrossEntropyLoss()

optimiser = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=NUM_EPOCHS, eta_min=1e-6
)

In [ ]:
# ─── 6. Training Loop ────────────────────────────────────────────────────────

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
           'val_recall_healthy': [], 'val_recall_diseased': []}

best_val_recall = 0.0
best_state = None


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = total_correct = total_n = 0
    # Per-class correct / total for recall
    per_class_correct = [0, 0]
    per_class_total   = [0, 0]

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False,
                                 desc='Train' if train else 'Val  '):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if train:
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()

            preds = logits.argmax(1)
            total_loss    += loss.item() * len(labels)
            total_correct += (preds == labels).sum().item()
            total_n       += len(labels)

            for c in range(2):
                mask = labels == c
                per_class_correct[c] += (preds[mask] == c).sum().item()
                per_class_total[c]   += mask.sum().item()

    recall = [per_class_correct[c] / max(per_class_total[c], 1) for c in range(2)]
    return total_loss / total_n, total_correct / total_n, recall


print(f'Training TinyBinaryCNN for {NUM_EPOCHS} epochs on {DEVICE}\n')
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc, _          = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_recall  = run_epoch(val_loader,   train=False)

    scheduler.step()

    # Primary metric: macro recall (we care equally about both classes)
    macro_recall = sum(va_recall) / 2

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_recall_healthy'].append(va_recall[0])
    history['val_recall_diseased'].append(va_recall[1])

    flag = ''
    if macro_recall > best_val_recall:
        best_val_recall = macro_recall
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, CHECKPOINT)
        flag = ' ★'

    print(
        f'Ep {epoch:02d}/{NUM_EPOCHS}  '
        f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  '
        f'va_loss={va_loss:.4f} va_acc={va_acc:.3f}  '
        f'R(H)={va_recall[0]:.3f} R(D)={va_recall[1]:.3f}  '
        f'macro_R={macro_recall:.3f}  '
        f'{time.time()-t0:.1f}s{flag}'
    )

print(f'\nBest macro recall: {best_val_recall:.4f}  — checkpoint saved to {CHECKPOINT}')

Training TinyBinaryCNN for 30 epochs on cuda



Ep 01/30  tr_loss=0.5402 tr_acc=0.720  va_loss=0.3733 va_acc=0.843  R(H)=0.713 R(D)=0.882  macro_R=0.798  275.5s ★


Ep 02/30  tr_loss=0.4159 tr_acc=0.814  va_loss=0.4817 va_acc=0.772  R(H)=0.923 R(D)=0.726  macro_R=0.824  273.6s ★


Ep 03/30  tr_loss=0.3362 tr_acc=0.858  va_loss=0.3618 va_acc=0.836  R(H)=0.969 R(D)=0.796  macro_R=0.882  275.1s ★


Ep 04/30  tr_loss=0.2670 tr_acc=0.893  va_loss=0.1768 va_acc=0.929  R(H)=0.845 R(D)=0.955  macro_R=0.900  272.0s ★


...

Ep 11/30  tr_loss=0.1490 tr_acc=0.942  va_loss=0.1122 va_acc=0.957  R(H)=0.948 R(D)=0.959  macro_R=0.954  275.7s ★


Ep 12/30  tr_loss=0.1411 tr_acc=0.949  va_loss=0.1650 va_acc=0.932  R(H)=0.976 R(D)=0.919  macro_R=0.947  277.2s


Ep 13/30  tr_loss=0.1376 tr_acc=0.949  va_loss=0.1578 va_acc=0.929  R(H)=0.979 R(D)=0.914  macro_R=0.947  273.4s


Ep 14/30  tr_loss=0.1424 tr_acc=0.949  va_loss=0.1156 va_acc=0.959  R(H)=0.975 R(D)=0.954  macro_R=0.965  274.3s ★


Train:  51%|█████     | 92/181 [01:54<01:46,  1.19s/it]

In [ ]:
# ─── 7. Training Curves ──────────────────────────────────────────────────────

epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('TinyBinaryCNN — Training History', fontsize=14, fontweight='bold')

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs, history['val_recall_healthy'],  label='Recall Healthy')
axes[2].plot(epochs, history['val_recall_diseased'], label='Recall Diseased')
axes[2].set_title('Validation Recall'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_curves_binary.png', dpi=120)
plt.show()

In [ ]:
# ─── 8. Final Evaluation on Test Set ────────────────────────────────────────

model.load_state_dict(best_state)
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

from sklearn.metrics import classification_report, confusion_matrix

print('Classification Report:')
print(classification_report(all_labels, all_preds,
                             target_names=['Healthy', 'Diseased']))

cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix (rows=true, cols=pred):')
print(f'             Healthy  Diseased')
print(f'Healthy      {cm[0,0]:6d}   {cm[0,1]:6d}')
print(f'Diseased     {cm[1,0]:6d}   {cm[1,1]:6d}')

# False-Negative rate (Diseased called Healthy) — the dangerous error
fnr = cm[1, 0] / cm[1].sum()
print(f'\nFalse-Negative Rate (disease missed): {fnr:.2%}')
print('(Target < 5% for deployment safety)')


In [ ]:
# ─── 9. ONNX Export ──────────────────────────────────────────────────────────
# ESP-DL quantisation pipeline takes an ONNX as input.
# The input shape must be (1, 3, 96, 96) — batch of 1, channels first.

model.load_state_dict(best_state)
model.eval()

dummy_input = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_OUT),
    opset_version=13,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input':  {0: 'batch'}, 'output': {0: 'batch'}},
)
print(f'ONNX exported → {ONNX_OUT}')

# Quick verification
import onnxruntime as ort
sess   = ort.InferenceSession(str(ONNX_OUT))
dummy_np = np.zeros((1, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
out    = sess.run(None, {'input': dummy_np})
print(f'ONNX inference shape: {out[0].shape}  ✓')

## 10. Quantisation for ESP-DL

Run these commands from your ESP-IDF environment after exporting the ONNX model.

```bash
# 1. Install ESP-DL python tools (once)
pip install idf-component-manager
pip install espdl  # or build from esp-dl/tools/

# 2. Quantise: INT8 symmetric, calibration from val set
python -m espdl.quantize \
    --onnx TinyBinaryCNN_v1.onnx \
    --calib_dir ./grape-leaf-disease-augmented-dataset/val \
    --output TinyBinaryCNN_v1_s8.espdl \
    --img_size 96 \
    --dtype int8

# 3. Copy to firmware models folder
cp TinyBinaryCNN_v1_s8.espdl <project>/models/
```

> **Note on exponent:** The firmware feeds images as `int8 = pixel − 128`, which corresponds to an exponent of **−7** (`2⁻⁷ = 1/128`). Make sure the quantisation tool is told about this via `--input_scale 0.0078125` or the equivalent exponent flag.

In [ ]:
# ─── 11. Deployment Sanity Check ────────────────────────────────────────────
# Simulate what the ESP32 actually receives: int8 = pixel - 128
# and verify the model still classifies correctly after that transform.

from PIL import Image

def simulate_esp32_preprocessing(image_path: str) -> torch.Tensor:
    """Replicate exactly what processor.py sends to the ESP32."""
    img = Image.open(image_path).convert('RGB').resize((96, 96), Image.Resampling.LANCZOS)
    arr = np.array(img, dtype=np.float32)           # [0, 255]
    arr_int8 = np.clip(np.round(arr - 128), -128, 127).astype(np.int8)
    # Reconstruct float for PyTorch: divide by 128 (exponent -7)
    arr_float = arr_int8.astype(np.float32) / 128.0  # [-1, ~1]
    tensor = torch.from_numpy(arr_float).permute(2, 0, 1).unsqueeze(0)  # NCHW
    return tensor


# ── Quick test with first val image ──────────────────────────────────────────
first_class = list((DATASET_ROOT / 'val').iterdir())[0]
first_img   = list(first_class.iterdir())[0]

model.eval()
with torch.no_grad():
    t_esp = simulate_esp32_preprocessing(str(first_img)).to(DEVICE)
    logits_esp = model(t_esp)
    pred_esp   = logits_esp.argmax(1).item()

label_map = {0: 'Healthy', 1: 'Diseased'}
print(f'Image: {first_img.name}')
print(f'True class folder: {first_class.name}')
print(f'ESP32-simulated prediction: {label_map[pred_esp]}')
print(f'Logits: {logits_esp.cpu().numpy()}')

In [ ]:
# ─── 12. Save Metadata ───────────────────────────────────────────────────────

meta = {
    'model': 'TinyBinaryCNN_v1',
    'task': 'binary_gate',
    'classes': {0: 'Healthy', 1: 'Diseased'},
    'healthy_folder': HEALTHY_FOLDER,
    'img_size': IMG_SIZE,
    'esp32_exponent': -7,
    'esp32_encoding': 'int8 = pixel - 128',
    'best_macro_recall': float(best_val_recall),
    'total_params': total_params,
    'checkpoint': str(CHECKPOINT),
    'onnx': str(ONNX_OUT),
}

with open('TinyBinaryCNN_v1_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Metadata saved.')
print(json.dumps(meta, indent=2))